# Build the corpus

Rebuilds the two corpus files the pipeline reads, fetching both through the MediaWiki API:

| File | Source | Selection |
|---|---|---|
| `documents/wikipedia_pages.json` | English Wikipedia | 33 fixed titles: the series, its titles and spin-offs, the consoles they released on, and the best-selling games list |
| `documents/mh_wiki_monsters.json` | [monsterhunterwiki.org](https://monsterhunterwiki.org) | every page in `Category:Large Monsters` and `Category:Small Monsters` |

Both files map page title to plain text. Run the notebook top to bottom. It takes a few minutes, mostly the
delay between Monster Hunter Wiki requests. Each save reports what changed against the existing file before
overwriting it.

Afterwards run `python -m rag.pipeline` from the repository root. Only chunks whose text changed are
re-embedded, and chunks the new corpus no longer produces are deleted from the store.

In [ ]:
import json
import re
import time
from pathlib import Path

import requests
import wikipediaapi
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# Works whether the kernel starts in notebooks/ or the repository root
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
OUT_DIR = ROOT / "documents"

# Wikimedia's API policy asks for a descriptive User-Agent with contact details
USER_AGENT = "MH-RAG/0.1 (https://github.com/DhLYa/RAG_Pipeline_Testing)"
HEADERS = {"User-Agent": USER_AGENT}

In [ ]:
def report_changes(pages: dict[str, str], path: Path) -> None:
    """Print which titles were added, removed or edited compared with the file at path."""
    with open(path, encoding="utf-8") as f:
        old_pages = json.load(f)

    added = sorted(set(pages) - set(old_pages))
    removed = sorted(set(old_pages) - set(pages))

    changed = []
    for title in pages:
        if title in old_pages and pages[title] != old_pages[title]:
            changed.append(title)

    print(f"vs existing {path.name}: {len(added)} added, {len(removed)} removed, {len(changed)} changed")
    if added:
        print(f"  added: {added[:10]}")
    if removed:
        print(f"  removed: {removed[:10]}")


def save_corpus(pages: dict[str, str], path: Path) -> None:
    """Report what changed against the existing file, then overwrite it."""
    if path.exists():
        report_changes(pages, path)

    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    total_chars = sum(len(text) for text in pages.values())
    print(f"Saved {len(pages)} pages, {total_chars:,} characters, to {path}")

## 1. Wikipedia

The title list is fixed rather than taken from a category, because the consoles and the best-selling games
list are not in `Category:Monster_Hunter`. Titles are kept exactly as first written (underscores and all):
each one becomes its chunks' `source` metadata and is hashed into their IDs, so renaming a title would force
every chunk from that page to be re-embedded. MediaWiki resolves them to the canonical articles.

Trailing sections holding citations and links are dropped by section title. `page.text` renders headings as
bare lines, so searching the text for `== References` markup, as the original notebook did, matches nothing.

In [ ]:
WIKIPEDIA_TITLES = [
    # Series, main titles and spin-offs
    "Monster_Hunter",
    "Monster_Hunter_(video_game)",
    "Monster_Hunter_2",
    "Monster_Hunter_4",
    "Monster_Hunter_Diary",
    "Monster_Hunter_Dynamic_Hunting",
    "Monster_Hunter_Freedom",
    "Monster_Hunter_Freedom_2",
    "Monster_Hunter_Freedom_Unite",
    "Monster_Hunter_Frontier_Z",
    "Monster_Hunter_Generations",
    "Monster_Hunter_Now",
    "Monster_Hunter_Outlanders",
    "Monster_Hunter_Portable_3rd",
    "Monster_Hunter_Rise",
    "Monster_Hunter_Stories",
    "Monster_Hunter_Stories_2: Wings_of_Ruin",
    "Monster_Hunter_Stories_3: Twisted_Reflection",
    "Monster_Hunter_Tri",
    "Monster_Hunter_Wilds",
    "Monster_Hunter_World: Iceborne",
    "Monster_Hunter: Legends_of_the_Guild",
    "Monster_Hunter: World",
    # Consoles the series released on
    "Nintendo_3DS",
    "Nintendo_Switch",
    "PlayStation",
    "PlayStation_(console)",
    "PlayStation_2",
    "PlayStation_3",
    "PlayStation_4",
    "PlayStation_Portable",
    "PlayStation_Vita",
    # Sales context
    "List_of_best-selling_video_games",
]

# Top-level sections that hold citations and links rather than prose
DROP_SECTIONS = {
    "References", "Notes", "Citations", "Footnotes", "Sources", "Bibliography",
    "External links", "See also", "Further reading",
}

wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language="en")


def article_text(page) -> str:
    """page.text without the DROP_SECTIONS. Assembled the same way wikipediaapi
    builds page.text, so the text that is kept is formatted identically."""
    text = page.summary
    if text:
        text += "\n\n"
    for section in page.sections:
        if section.title not in DROP_SECTIONS:
            text += section.full_text(level=2)
    return text.strip()

In [ ]:
wikipedia_pages, missing = {}, []
for title in tqdm(WIKIPEDIA_TITLES, unit="page"):
    page = wiki.page(title)
    if page.exists():
        wikipedia_pages[title] = article_text(page)
    else:
        missing.append(title)

# A missing title means an article was renamed or deleted; fix the list rather than save a smaller corpus
if missing:
    raise RuntimeError(f"Wikipedia pages not found: {missing}")

save_corpus(wikipedia_pages, OUT_DIR / "wikipedia_pages.json")

## 2. Monster Hunter Wiki

Pages are requested as rendered HTML and reduced to prose:

- **Infobox first.** `table.monster-game-info` is read into label-value lines (classification, elements,
  weaknesses, ...) before the remaining tables are stripped. Element and status values are rendered as icons
  whose names live on the parent link, so they are recovered from the link rather than the image.
- **Headings only with content.** A heading is kept only when prose follows it. Sections whose content was in
  the stripped tables or galleries would otherwise leave runs of bare headings that become empty chunks.
- **Noise removed.** Navigation bars, galleries, figures, reference lists and MediaWiki citation errors.
- **Stubs skipped.** Pages whose prose, excluding the infobox, is under `MIN_BODY_CHARS`, or whose only content
  is the legend explaining the stat tables.

In [ ]:
MH_API = "https://monsterhunterwiki.org/api.php"
CATEGORIES = ["Category:Large Monsters", "Category:Small Monsters"]

MIN_PAGE_CHARS = 400
MIN_BODY_CHARS = 300          # prose alone, excluding the infobox
REQUEST_DELAY = 0.3           # seconds between page requests

INFOBOX_SELECTOR = "table.monster-game-info"

# This wiki uses <h1> for section headings, not <h2>
HEADING_TAGS = {"h1", "h2", "h3", "h4", "h5", "h6"}
BLOCK_TAGS = ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "dd", "dt", "blockquote"]

DROP_SELECTORS = [
    "table",
    "div.navbarheader",
    "div.navbar-green",
    "div.toc",
    "sup.reference",
    "span.mw-editsection",
    "figure",
    "figcaption",
    "div.thumb",
    "div.gallery",
    "div.mw-references-wrap",
    ".mw-ext-cite-error",      # "Cite error: <ref> tags exist for..."
    "span.error",
]

SKIP_NS = ("File:", "Help:", "Category:", "Template:", "Special:", "Talk:")

# Pages whose only content is the legend explaining the stat tables we strip.
# Their monster data lived in those tables, so nothing usable remains.
BOILERPLATE_MARKERS = (
    "The following information is for",
    "Part - the name of the part",
    "the effectiveness of Cutting, Blunt, and Shot",
)

In [ ]:
def category_members(category, namespace=0):
    """All article titles in a category. The API returns them in batches, each
    carrying a "continue" token for the next until the category is exhausted."""
    titles = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": category,
        "cmlimit": "max",
        "cmnamespace": namespace,
        "format": "json",
    }
    while True:
        r = requests.get(MH_API, params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()

        for member in data["query"]["categorymembers"]:
            titles.append(member["title"])

        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.2)
    return titles


# A page in both categories is fetched once
unique_titles = set()
for category in CATEGORIES:
    titles = category_members(category)
    print(f"{category}: {len(titles)} pages")
    unique_titles.update(titles)

mh_titles = sorted(unique_titles)
print(f"Unique total: {len(mh_titles)}")

### Cleaning a page

In [ ]:
def tidy(text):
    """Normalise the whitespace and punctuation spacing left behind by HTML extraction."""
    text = re.sub(r"\s+", " ", text).strip()          # collapse runs of whitespace
    text = re.sub(r"\s+([.,;:!?)\]])", r"\1", text)   # "word ." -> "word."
    text = re.sub(r"([(\[])\s+", r"\1", text)         # "( word" -> "(word"
    text = re.sub(r",\s*\)", ")", text)               # "(Dragon,)" -> "(Dragon)"
    text = re.sub(r"\(\s*\)", "", text)               # parens left empty by icons
    text = re.sub(r"\s{2,}", " ", text)               # spaces left by the removals above
    return text.strip(" ,;")


def cell_text(cell):
    return tidy(cell.get_text(" ", strip=True))


def table_to_text(table):
    """Turn a wikitable into "label: value" lines. Handles two layouts:

    - vertical: each row holds its own label and value, <th>Label</th><td>Value</td>
    - horizontal: a row of labels, followed by a row of values
    """
    lines = []
    pending_labels = None    # a header row still waiting for its row of values

    for row in table.find_all("tr"):
        headers = row.find_all("th", recursive=False)
        cells = row.find_all("td", recursive=False)

        if headers and cells:
            labels = [cell_text(h) for h in headers]
            values = [cell_text(c) for c in cells]
            pending_labels = None
        elif headers:
            pending_labels = [cell_text(h) for h in headers]
            continue
        elif cells and pending_labels:
            labels = pending_labels
            values = [cell_text(c) for c in cells]
            pending_labels = None
        else:
            continue

        for label, value in zip(labels, values):
            if label and value:
                lines.append(f"{label}: {value}")

    return "\n".join(lines)


def icon_label(img):
    """Name of an element or status icon, taken from the link wrapping it.
    None for icons that aren't values, such as renders and help links."""
    link = img.find_parent("a")
    if link is None:
        return None

    href = link.get("href", "")
    for namespace in SKIP_NS:
        if href.startswith(f"/wiki/{namespace}"):
            return None

    label = link.get("title") or href.rsplit("/", 1)[-1].replace("_", " ")
    return re.sub(r"\s+Element$", "", label)    # "Fire Element" -> "Fire"


def infobox_text(soup):
    """The monster infobox as "label: value" lines, or "" if the page has none."""
    table = soup.select_one(INFOBOX_SELECTOR)
    if table is None:
        return ""

    for footnote in table.find_all("sup"):
        footnote.decompose()

    # Element and status values are icons; swap each for its name so it survives as text
    for img in table.find_all("img"):
        label = icon_label(img)
        if label:
            img.replace_with(f" {label}, ")
        else:
            img.replace_with("")

    return table_to_text(table)


def body_text(soup):
    """Prose blocks, one per line, keeping a heading only when real content follows it.

    Sections whose content was in stripped tables or galleries otherwise leave
    long runs of consecutive headings with nothing beneath them.
    """
    blocks = []
    pending_heading = None    # most recent heading, not yet followed by content

    for element in soup.find_all(BLOCK_TAGS):
        text = tidy(element.get_text(" ", strip=True))
        if not text:
            continue

        if element.name in HEADING_TAGS:
            pending_heading = text    # replaces any earlier heading that had no content
            continue

        if pending_heading:
            blocks.append(pending_heading)
            pending_heading = None
        blocks.append(text)

    return "\n".join(blocks)


def clean_page(html):
    """Infobox lines followed by the prose, or None if the page is a stub."""
    soup = BeautifulSoup(html, "html.parser")

    # The infobox is a table, so it has to be read before the tables are stripped
    facts = infobox_text(soup)
    for selector in DROP_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()
    body = body_text(soup)

    # Judge the stub check on prose alone: a page with a populated infobox but
    # no descriptive text is still a stub, and checking the combined length
    # would let it through on the strength of the infobox.
    if any(marker in body for marker in BOILERPLATE_MARKERS):
        return None
    if len(body) < MIN_BODY_CHARS:
        return None

    if facts:
        return f"{facts}\n\n{body}".strip()
    return body.strip()

### Fetching

In [ ]:
def fetch_page_html(title):
    """Rendered HTML of a page, or None if the wiki can't parse it."""
    params = {
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "formatversion": 2,
    }
    r = requests.get(MH_API, params=params, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        return None

    data = r.json()
    if "parse" not in data:
        return None
    return data["parse"]["text"]


def fetch_page_text(title):
    html = fetch_page_html(title)
    if html is None:
        return None
    return clean_page(html)


def fetch_all(titles):
    """Fetch and clean every title, keeping pages of at least MIN_PAGE_CHARS."""
    pages, failed = {}, []
    total_chars = 0

    bar = tqdm(titles, unit="page", desc="fetching")
    for title in bar:
        try:
            text = fetch_page_text(title)
        except Exception as e:
            failed.append((title, type(e).__name__))
            continue

        if text and len(text) >= MIN_PAGE_CHARS:
            pages[title] = text
            total_chars += len(text)
        else:
            failed.append((title, "stub, empty, or too short"))

        bar.set_postfix_str(f"{title[:24]} | {len(pages)} kept | {total_chars:,} chars")
        time.sleep(REQUEST_DELAY)
    bar.close()

    if failed:
        print(f"\n{len(failed)} failed or skipped, e.g. {failed[:5]}")
    return pages

Optional: preview a few cleaned pages before committing to the full fetch.

In [ ]:
preview = fetch_all(mh_titles[:3])
for title, text in preview.items():
    print(f"\n--- {title} ({len(text)} chars) ---")
    print(text[:700])

In [ ]:
mh_pages = fetch_all(mh_titles)
print(f"\nKept {len(mh_pages)} of {len(mh_titles)} pages.")

save_corpus(mh_pages, OUT_DIR / "mh_wiki_monsters.json")

## 3. Checks

In [ ]:
# Pages without an infobox lose the structured attributes (classification, elements, weaknesses)
no_infobox = []
for title, text in mh_pages.items():
    if "Classification:" not in text:
        no_infobox.append(title)

print(f"{len(no_infobox)} of {len(mh_pages)} Monster Hunter Wiki pages lack an infobox")
print(no_infobox[:15])

# The Wikipedia text should no longer contain the sections dropped in part 1
reference_heading = re.compile(r"^(References|External links)$", flags=re.MULTILINE)
leftover = []
for title, text in wikipedia_pages.items():
    if reference_heading.search(text):
        leftover.append(title)

print(f"Wikipedia pages still containing reference sections: {leftover or 'none'}")